# 02 — Models & Providers

## Objectives

- Master the v2 model-string grammar: `<provider>:<model>`
- Pass per-run and per-agent model settings
- Make an agent resilient with `FallbackModel`
- Compare token usage and cost across models — and read it in Logfire's LLMs view

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

## Model strings in v2

The first argument of `Agent` is a **prefixed** string — the prefix picks the provider *and* the wire API:

| Prefix | Meaning |
|---|---|
| `openai:` | OpenAI **Responses API** (the modern default — reasoning + tools work together) |
| `openai-chat:` | OpenAI Chat Completions (legacy; needed for a few old settings) |
| `openai-codex:` | your ChatGPT/Codex **subscription** instead of an API key |
| `anthropic:` | Claude models |
| `google:` | Gemini models |
| `xai:` | xAI/Grok models |

A bare name like `Agent('gpt-6.1-sol')` raises `UserError` in v2 — if you hit that, you've
copy-pasted from an old tutorial.

In [ ]:
from pydantic_ai import Agent

agent = Agent(MODEL, instructions="Answer with one short sentence.")

result = await agent.run("In one sentence: why do agent frameworks exist at all?")
print(result.output)

> ✅ **Checkpoint** — a plain answer from your `MODEL` string. Swap `MODEL` in `.env`
> (say to `anthropic:claude-...`) and re-run this cell — the same notebook works unchanged.

## Model settings

`ModelSettings` is a `TypedDict`, so a plain dict works. Two things worth knowing:

- Settings can be passed **per run** or fixed **on the agent**.
- Classic knobs like `temperature` are silently ignored by some modern reasoning models —
  for the gpt-6 family you steer with `openai_reasoning_effort` instead (`'low'` → fast and cheap,
  `'high'` → slow and thorough).

In [ ]:
# Per-run settings — just for this one call:
result_low = await agent.run(
    "In one word: is water wet?",
    model_settings={"openai_reasoning_effort": "low"},
)
print("per-run    :", result_low.output)

# Per-agent settings — the default for every future run of this agent:
agent_fast = Agent(
    MODEL,
    instructions="Answer with one short sentence.",
    model_settings={"openai_reasoning_effort": "low"},
)
r = await agent_fast.run("Why is the sky blue?")
print("per-agent  :", r.output)

## FallbackModel — automatic failover

Rate limits and outages happen. Wrap your models in `FallbackModel` and pydantic-ai retries the
run on the next model when the first one raises an API error — one result either way.

In [ ]:
from pydantic_ai.models.fallback import FallbackModel

resilient = Agent(
    FallbackModel(MODEL, "openai:gpt-5.6-sol"),  # primary, then fallback
    instructions="Say something reassuring about error handling. One sentence.",
)

r = await resilient.run("What happens if you fail?")
print(r.output)

> ✅ **Checkpoint** — same shape as any agent from outside. If you want to *see* it fire,
> temporarily set the primary to a nonsense model name, re-run, and watch the fallback take
> over in the Logfire trace.

## Compare models — the meter, side by side

Same prompt, two models, `result.usage` tells you what each charged.

In [ ]:
prompt = 'Explain what a "token" is, in exactly one sentence.'

for label, model_string in {
    "primary (MODEL)": MODEL,
    "second opinion  ": "openai:gpt-5.6-sol",  # swap in any model your key supports
}.items():
    comparison_agent = Agent(model_string, instructions="Answer with exactly one sentence.")
    r = await comparison_agent.run(prompt)
    print(f"{label} | in={r.usage.input_tokens} out={r.usage.output_tokens} cost={r.usage.cost}")
    print(f"{'':17}| {r.output}")

> ✅ **Checkpoint** — two rows, two different price tags for the same question.
> 🔭 In Logfire's **LLMs view** this comparison is automatic: latency, tokens/sec, error rate
> and cost per model, across all your notebook runs.

## 🏋️ Exercise — the model shoot-out

Run one prompt through **three** different model strings (e.g. your `MODEL`,
`openai:gpt-5.6-sol`, `openai:gpt-5.5` — or mix providers if you have keys). Record
`input_tokens`, `output_tokens` and `cost` for each, then answer: which is the cheapest
*per answer*, and was the quality difference worth it? (This is the real production
decision pydantic-ai's `result.usage.cost` exists for.)

Your take: [`../solutions/02_compare_models.ipynb`](../solutions/02_compare_models.ipynb).

## Next

→ **[03_structured_outputs.ipynb](03_structured_outputs.ipynb)** — the feature that makes
pydantic-ai *pydantic*: validated Python objects instead of strings.